In [4]:
# Cell 1 - Imports
# We bring in all the tools we need
# Run this first, every time you open the notebook

import os
import json
import numpy as np
import faiss
import wikipediaapi
from sentence_transformers import SentenceTransformer

print("All imports successful ✓")

c:\Users\titir\routellm\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


All imports successful ✓


In [5]:
# Cell 2 - Load the embedding model
# This model converts text into vectors (lists of 384 numbers)
# First run downloads ~80MB from the internet — wait for it

print("Loading embedding model...")
model = SentenceTransformer("all-MiniLM-L6-v2")
print(f"✓ Model loaded")
print(f"  Vector size this model produces: {model.get_embedding_dimension()} dimensions")

Loading embedding model...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3920.41it/s]


✓ Model loaded
  Vector size this model produces: 384 dimensions


In [6]:
# Cell 3 - See what a vector actually looks like
# Let's embed two similar sentences and one different one
# and see how the numbers reflect meaning

sentences = [
    "plasma instability in nuclear fusion",  # physics
    "tokamak reactor magnetic confinement",  # physics - similar
    "how to write a Python function"         # code - different
]

# Convert all three to vectors
vectors = model.encode(sentences)

print(f"Shape: {vectors.shape}")
print(f"→ {len(sentences)} sentences, each became {vectors.shape[1]} numbers\n")

# Show the first 8 numbers of each vector (full 384 is too long to read)
for i, sentence in enumerate(sentences):
    print(f"'{sentence}'")
    print(f"  First 8 numbers: {vectors[i][:8].round(3)}\n")

# Now calculate distance between them
# Small distance = similar meaning
# Large distance = different meaning
import numpy as np

def distance(v1, v2):
    return np.sqrt(np.sum((v1 - v2) ** 2))

d_physics = distance(vectors[0], vectors[1])
d_different = distance(vectors[0], vectors[2])

print(f"Distance: physics vs physics = {d_physics:.3f}")
print(f"Distance: physics vs code   = {d_different:.3f}")
print(f"\n→ Smaller number = more similar meaning")

Shape: (3, 384)
→ 3 sentences, each became 384 numbers

'plasma instability in nuclear fusion'
  First 8 numbers: [-0.055 -0.012  0.021  0.089 -0.061 -0.011 -0.019  0.017]

'tokamak reactor magnetic confinement'
  First 8 numbers: [-0.052  0.046 -0.082  0.076  0.024  0.064 -0.052 -0.009]

'how to write a Python function'
  First 8 numbers: [-0.003  0.086 -0.035  0.025 -0.087 -0.069  0.075  0.06 ]

Distance: physics vs physics = 0.945
Distance: physics vs code   = 1.452

→ Smaller number = more similar meaning


In [7]:
# Cell 4 - Fetch a real Wikipedia article
# This is our raw data source for the knowledge bases

import wikipediaapi

wiki = wikipediaapi.Wikipedia(
    language="en",
    user_agent="RouteLLM/0.1 (research project)"
)

# Fetch one article
topic = "Plasma (physics)"
page = wiki.page(topic)

print(f"Topic: {page.title}")
print(f"Exists: {page.exists()}")
print(f"Full article length: {len(page.text)} characters")
print(f"\nFirst 500 characters:")
print(page.text[:500])

Topic: Plasma (physics)
Exists: True
Full article length: 30979 characters

First 500 characters:
Plasma is a state of matter that results from one of the other three states (often, the gaseous one) having undergone an appreciable degree of ionization. It thus consists of a significant portion of charged particles (ions and/or electrons). While rarely encountered on Earth, it is estimated that 99.9% of all ordinary matter in the universe is plasma. Stars are almost pure balls of plasma, and plasma dominates the rarefied intracluster medium and intergalactic medium. Plasma can be artificially


In [8]:
# Cell 5 - Chunking
# Split a long article into smaller pieces
# Each piece gets its own 384D vector

def chunk_text(text, chunk_size=500, overlap=50):
    """
    Split text into chunks of ~chunk_size characters
    overlap = how many characters to share between chunks
    Why overlap? So we don't lose context at chunk boundaries
    """
    chunks = []
    start = 0
    
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        # Move forward by chunk_size minus overlap
        start += chunk_size - overlap
    
    return chunks

# Test it on our plasma article
article_text = page.text
chunks = chunk_text(article_text)

print(f"Article length: {len(article_text)} characters")
print(f"Number of chunks: {len(chunks)}")
print(f"Average chunk size: {sum(len(c) for c in chunks) // len(chunks)} characters")
print(f"\nFirst chunk preview:")
print(chunks[0])
print(f"\n--- chunk boundary ---\n")
print(f"Second chunk preview (notice overlap with first):")
print(chunks[1][:200])

Article length: 30979 characters
Number of chunks: 69
Average chunk size: 498 characters

First chunk preview:
Plasma is a state of matter that results from one of the other three states (often, the gaseous one) having undergone an appreciable degree of ionization. It thus consists of a significant portion of charged particles (ions and/or electrons). While rarely encountered on Earth, it is estimated that 99.9% of all ordinary matter in the universe is plasma. Stars are almost pure balls of plasma, and plasma dominates the rarefied intracluster medium and intergalactic medium. Plasma can be artificially

--- chunk boundary ---

Second chunk preview (notice overlap with first):
d intergalactic medium. Plasma can be artificially generated, for example, by heating a neutral gas or subjecting it to a strong electromagnetic field.
The presence of charged particles makes plasma e


In [9]:
# Cell 6 - Embed all chunks and build FAISS index
# This is the heart of the knowledge base

print(f"Embedding {len(chunks)} chunks...")
chunk_embeddings = model.encode(chunks, show_progress_bar=True)

print(f"\nEmbeddings shape: {chunk_embeddings.shape}")
print(f"→ {chunk_embeddings.shape[0]} chunks, each = {chunk_embeddings.shape[1]} dimensions")

# Build FAISS index
dimension = chunk_embeddings.shape[1]  # 384
index = faiss.IndexFlatL2(dimension)

# Add all chunk vectors to the index
index.add(np.array(chunk_embeddings, dtype=np.float32))

print(f"\n✓ FAISS index built")
print(f"  Vectors stored: {index.ntotal}")
print(f"  Each vector: {dimension} dimensions")

Embedding 69 chunks...


Batches: 100%|██████████| 3/3 [00:01<00:00,  2.99it/s]


Embeddings shape: (69, 384)
→ 69 chunks, each = 384 dimensions

✓ FAISS index built
  Vectors stored: 69
  Each vector: 384 dimensions


In [10]:
# Cell 7 - Search the knowledge base
# Ask a question, find the most relevant chunks

def search_kb(question, index, chunks, model, top_k=3):
    """
    Given a question:
    1. Convert question to a 384D vector
    2. Ask FAISS: which stored vectors are closest?
    3. Return the matching chunks
    """
    # Step 1: embed the question
    question_vector = model.encode([question])
    
    # Step 2: search FAISS for top_k nearest vectors
    # D = distances, I = indices of nearest chunks
    D, I = index.search(
        np.array(question_vector, dtype=np.float32), 
        top_k
    )
    
    # Step 3: return the actual text chunks
    results = []
    for rank, (dist, idx) in enumerate(zip(D[0], I[0])):
        results.append({
            "rank": rank + 1,
            "distance": round(float(dist), 3),
            "chunk": chunks[idx]
        })
    return results

# Test it with two very different questions
questions = [
    "how is plasma generated artificially?",
    "what is the temperature of plasma in stars?"
]

for question in questions:
    print(f"\nQuestion: '{question}'")
    print("-" * 50)
    results = search_kb(question, index, chunks, model)
    
    for r in results:
        print(f"\nRank {r['rank']} (distance: {r['distance']})")
        print(r['chunk'][:300])
        print("...")


Question: 'how is plasma generated artificially?'
--------------------------------------------------

Rank 1 (distance: 0.343)
(the magnetic field is too weak to trap the particles in orbits but may generate Lorentz forces)

Generation of artificial plasma
Just like the many uses of plasma, there are several means for its generation. However, one principle is common to all of them: there must be energy input to produce and 
...

Rank 2 (distance: 0.607)
n close binary star systems. Plasma is associated with ejection of material in astrophysical jets, which have been observed with accreting black holes or in active galaxies like M87's jet that possibly extends out to 5,000 light-years.

Artificial plasmas
Most artificial plasmas are generated by the
...

Rank 3 (distance: 0.629)
d intergalactic medium. Plasma can be artificially generated, for example, by heating a neutral gas or subjecting it to a strong electromagnetic field.
The presence of charged particles makes plasma electricall

In [11]:
# Cell 9 - Debug: see what Ollama actually returned

import requests

test_response = requests.post(
    "http://localhost:11434/api/generate",
    json={
        "model": "qwen3:8b",
        "prompt": "Say hello in one word.",
        "stream": False
    }
)

# Print the full raw response
print("Status code:", test_response.status_code)
print("Full response:")
print(test_response.json())

Status code: 200
Full response:
{'model': 'qwen3:8b', 'created_at': '2026-06-16T04:47:47.7382368Z', 'response': 'Hello.', 'thinking': 'Okay, the user wants me to say hello in one word. Let me think. The most common greeting is "Hello," but maybe they want something more creative. Let me consider other possibilities. "Hi" is another common one, but maybe too short. What about "Greetings"? That\'s a bit more formal. Or "Salutations"? That\'s more elaborate. Wait, the user specified "one word," so I need to make sure it\'s a single word. "Hello" is definitely the simplest and most direct. It\'s widely recognized and universally understood. Maybe they just want the straightforward answer. Let me check if there\'s any other single-word greeting that\'s commonly used. "Hey" is another option, but it\'s more casual. "Hola" is Spanish for hello, but that\'s not English. The user didn\'t specify the language, but since the instruction is in English, probably expects an English word. So "Hello" 

In [12]:
import requests
import json

def ask_ollama(prompt, model="qwen3:8b"):
    """
    Send a prompt to Ollama and get a response.
    Ollama runs locally on port 11434.
    """
    try:
        response = requests.post(
            "http://localhost:11434/api/generate",
            json={
                "model": model,
                "prompt": "/no_think\n" + prompt,
                "stream": False  # wait for full response
            }
        )
        response.raise_for_status()
        return response.json()["response"]
    except Exception as e:
        return f"Error connecting to Ollama: {e}"


def rewrite_query(question, ollama_model):
    """Rewrites the question to be more specific for KB search."""
    prompt = f"""Rewrite this question to be more specific and technical for searching a science or coding knowledge base. Return ONLY the rewritten question.
    
Original: {question}
Rewritten:"""
    return ask_ollama(prompt, model=ollama_model).strip()


def rag_answer(question, index, chunks, model, ollama_model="qwen3:8b"):
    """
    Full RAG pipeline:
    1. Rewrite query for better retrieval
    2. Search KB for relevant chunks using the rewritten query
    3. Build a prompt with those chunks as context
    4. Send to Ollama and return the final answer
    """
    # Step 0: Rewrite the question first
    better_question = rewrite_query(question, ollama_model)
    print(f"Original Query:  {question}")
    print(f"Rewritten Query: {better_question}\n")

    # Step 1: Retrieve relevant chunks using the BETTER question
    # (Assumes search_kb is defined in a previous cell)
    results = search_kb(better_question, index, chunks, model, top_k=3)
    
    # Step 2: Build context from retrieved chunks
    context = "\n\n---\n\n".join([r["chunk"] for r in results])
    
    # Step 3: Build the final LLM prompt
    llm_prompt = f"""You are a helpful assistant. Answer the question using 
ONLY the context provided below. If the answer isn't in the context, 
say "I don't have enough information about that."

Context:
{context}

Question: {question}

Answer:"""
    
    # Step 4: Get answer from Ollama
    print(f"Retrieved {len(results)} chunks (distances: {[round(r['distance'], 4) for r in results]})")
    print(f"Generating answer...\n")
    
    answer = ask_ollama(llm_prompt, ollama_model)
    return answer


# Test it
question = "How is plasma generated artificially?"
# Note: Ensure 'index', 'chunks', and 'model' (embedding model) are defined upstream.
answer = rag_answer(question, index, chunks, model)

print("Answer:")
print(answer)

Original Query:  How is plasma generated artificially?
Rewritten Query: What are the technical methods and physical mechanisms involved in the artificial generation of plasma, including processes such as electrical discharge, laser-induced ionization, or magnetic confinement in controlled laboratory environments?

Retrieved 3 chunks (distances: [0.455, 0.69, 0.714])
Generating answer...

Answer:
Plasma is generated artificially through several methods, all requiring energy input. Key methods include:  
1. **Electric current application**: Passing an electric current across a dielectric gas or fluid (non-conductive material).  
2. **Capacitive discharge**: Using RF power (e.g., 13.56 MHz) between electrodes, often with noble gases like helium or argon for stabilization.  
3. **Piezoelectric direct discharge**: Generating nonthermal plasma at the high side of a piezoelectric setup.  
4. **Plasma bullets**: Fast propagating guided ionization waves that produce plasma jets.  

These method

In [14]:
# Cell 9 - Build the Code/Math KB
# Same pipeline as physics, different topics

CODE_MATH_TOPICS = [
    "Python (programming language)",
    "Algorithm",
    "Data structure",
    "Recursion",
    "Object-oriented programming",
    "Linear algebra",
    "Calculus",
    "Differential equation",
    "Matrix multiplication",
    "Gradient descent"
]
# Define fetch_wikipedia as a reusable function
def fetch_wikipedia(topic):
    """Fetch a Wikipedia article and return its text"""
    wiki = wikipediaapi.Wikipedia(
        language="en",
        user_agent="RouteLLM/0.1 (research project)"
    )
    page = wiki.page(topic)
    if page.exists():
        text = page.text[:3000]
        print(f"  ✓ Fetched: {topic}")
        return {"topic": topic, "text": text}
    else:
        print(f"  ✗ Not found: {topic}")
        return None

print("Fetching code/math articles...")
code_math_docs = []
for topic in CODE_MATH_TOPICS:
    doc = fetch_wikipedia(topic)
    if doc:
        code_math_docs.append(doc)

print(f"\nFetched {len(code_math_docs)} articles")

# Chunk all articles
code_math_chunks = []
for doc in code_math_docs:
    chunks_for_doc = chunk_text(doc["text"])
    code_math_chunks.extend(chunks_for_doc)

print(f"Total chunks: {len(code_math_chunks)}")

# Embed all chunks
print("\nEmbedding chunks...")
code_math_embeddings = model.encode(
    code_math_chunks, 
    show_progress_bar=True
)

# Build second FAISS index
dimension = code_math_embeddings.shape[1]
code_math_index = faiss.IndexFlatL2(dimension)
code_math_index.add(
    np.array(code_math_embeddings, dtype=np.float32)
)

print(f"\n✓ Code/Math KB built")
print(f"  Chunks: {code_math_index.ntotal}")
print(f"  Dimensions: {dimension}")

Fetching code/math articles...
  ✓ Fetched: Python (programming language)
  ✓ Fetched: Algorithm
  ✓ Fetched: Data structure
  ✓ Fetched: Recursion
  ✓ Fetched: Object-oriented programming
  ✓ Fetched: Linear algebra
  ✓ Fetched: Calculus
  ✓ Fetched: Differential equation
  ✓ Fetched: Matrix multiplication
  ✓ Fetched: Gradient descent

Fetched 10 articles
Total chunks: 70

Embedding chunks...


Batches: 100%|██████████| 3/3 [00:01<00:00,  2.67it/s]


✓ Code/Math KB built
  Chunks: 70
  Dimensions: 384


In [15]:
# Cell 10 - The Router
# Decides which KB to search based on the question
# Then runs the full RAG pipeline on the right specialist

# --- STEP 1: Define the router ---

PHYSICS_KEYWORDS = [
    "plasma", "fusion", "quantum", "electromagnetic",
    "thermodynamics", "relativity", "particle", "nuclear",
    "magnetic", "electric field", "photon", "neutron",
    "reactor", "tokamak", "energy", "wave", "radiation"
]

CODE_MATH_KEYWORDS = [
    "python", "code", "algorithm", "function", "loop",
    "recursion", "class", "object", "programming", "debug",
    "matrix", "calculus", "derivative", "integral", "gradient",
    "linear algebra", "equation", "vector", "eigenvalue"
]

def route(question):
    """
    Keyword-based router
    Counts how many domain keywords appear in the question
    Routes to whichever domain scores higher
    """
    question_lower = question.lower()
    
    physics_score = sum(
        1 for kw in PHYSICS_KEYWORDS 
        if kw in question_lower
    )
    
    code_math_score = sum(
        1 for kw in CODE_MATH_KEYWORDS 
        if kw in question_lower
    )
    
    print(f"  Physics score:    {physics_score}")
    print(f"  Code/Math score:  {code_math_score}")
    
    if code_math_score > physics_score:
        return "code_math"
    else:
        return "physics"  # default to physics if tie


# --- STEP 2: Combined ask() function ---

def ask(question):
    """
    Full RouteLLM pipeline:
    1. Router decides which specialist to use
    2. Search the right KB
    3. Generate answer from retrieved chunks
    """
    print(f"\nQuestion: {question}")
    print(f"Routing...")
    
    domain = route(question)
    print(f"  → Routed to: {domain}\n")
    
    if domain == "physics":
        results = search_kb(question, index, chunks, model)
    else:
        results = search_kb(
            question, 
            code_math_index, 
            code_math_chunks, 
            model
        )
    
    # Build context from retrieved chunks
    context = "\n\n---\n\n".join([r["chunk"] for r in results])
    
    # Build prompt
    prompt = f"""You are a helpful assistant. Answer the question 
using ONLY the context provided below. If the answer isn't in 
the context, say "I don't have enough information about that."

Context:
{context}

Question: {question}

Answer:"""
    
    answer = ask_ollama(prompt)
    return answer


# --- STEP 3: Test the router with two questions ---

q1 = "How is plasma generated artificially?"
q2 = "What is gradient descent in machine learning?"

print("=" * 50)
answer1 = ask(q1)
print("Answer:", answer1)

print("=" * 50)
answer2 = ask(q2)
print("Answer:", answer2)


Question: How is plasma generated artificially?
Routing...
  Physics score:    1
  Code/Math score:  0
  → Routed to: physics

Answer: Artificial plasma is generated by applying electric and/or magnetic fields through a gas, or by heating a neutral gas. These methods provide the necessary energy input to create and sustain the plasma state.

Question: What is gradient descent in machine learning?
Routing...
  Physics score:    0
  Code/Math score:  1
  → Routed to: code_math

Answer: Gradient descent is a first-order iterative optimization algorithm used in machine learning to minimize the cost or loss function by iteratively adjusting parameters in the direction opposite to the gradient. It is particularly useful for training models by reducing error through gradient steps and is foundational in minimizing the loss function during machine learning tasks. A simple extension of it, stochastic gradient descent, is widely used for training deep networks.
